In [1]:
pip install transformers torch easyocr pillow sentencepiece

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import CLIPProcessor, CLIPModel
from transformers import MarianMTModel, MarianTokenizer
import easyocr
from PIL import Image
import torch

# ===== PATH =====
cache_path = "D:/huggingface"

# ===== CLIP =====
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir=cache_path
)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir=cache_path
)

# ===== OCR =====
reader = easyocr.Reader(
    ['en', 'vi'],
    model_storage_directory="D:/easyocr"
)

# ===== TRANSLATE =====
model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = MarianTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_path
)

translator = MarianMTModel.from_pretrained(
    model_name,
    cache_dir=cache_path
)

# ===== PIPELINE FUNCTION =====
def process_image(image_path):
    # 1. Load image
    image = Image.open(image_path).convert("RGB")

    # 2. OCR
    results = reader.readtext(image_path)
    text = " ".join([res[1] for res in results])

    print("OCR Text:", text)

    # 3. Translate EN -> VI
    if text.strip() != "":
        inputs = tokenizer([text], return_tensors="pt", padding=True)
        translated = translator.generate(**inputs)
        vi_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    else:
        vi_text = ""

    print("Translated:", vi_text)

    # 4. CLIP (test với label)
    labels = ["a photo", "a document", "a meme", "a screenshot"]

    inputs = clip_processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    )

    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

    print("CLIP Prediction:")
    for label, p in zip(labels, probs[0]):
        print(f"{label}: {p.item():.4f}")

    return {
        "ocr": text,
        "translated": vi_text,
        "clip_probs": probs
    }

# ===== TEST =====
result = process_image("test.jpg")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'test.jpg'

In [ ]:
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir="D:/huggingface",
)

In [3]:
from transformers import CLIPModel

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir="D:/huggingface"
)

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors
Retrying in 1s [Retry 1/5].


OSError: Can't load the model for 'openai/clip-vit-base-patch32'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'openai/clip-vit-base-patch32' is the correct path to a directory containing a file named pytorch_model.bin.

In [4]:
model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir="D:/huggingface",
    force_download=True
)

OSError: Force download failed due to the above error.

In [5]:
force_download=True

In [2]:
from transformers import CLIPModel, CLIPProcessor
model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    cache_dir="D:/huggingface"
)

config.json: 0.00B [00:00, ?B/s]

C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [6]:
from PIL import Image

def process_image(image_path):
    img = Image.open(image_path)
    return img

In [18]:
result = process_image("harvard_0.jpg")

In [19]:
pip install easyocr

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
